In [ ]:
from pathlib import Path

import os
import json
import math
import re
import subprocess
import warnings

import numpy as np
import pandas as pd
import nibabel as nib

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torchvision.models import (
    inception_v3,
    Inception_V3_Weights
)

from scipy import linalg
from scipy.spatial.distance import cdist

from sklearn.preprocessing import StandardScaler

In [ ]:
# ============================================================
# Global configuration
# ============================================================

PROJECT_ROOT = Path.cwd()

EVALUATION_ROOT = (
    PROJECT_ROOT
    / "evaluation_200"
)

REAL_DIR = (
    EVALUATION_ROOT
    / "real"
)

CONDITIONS_DIR = (
    EVALUATION_ROOT
    / "conditions"
)

COHORT_JSON = (
    CONDITIONS_DIR
    / "evaluation_subjects_200.json"
)

DATA_SPLIT_JSON = (
    PROJECT_ROOT
    / "data_split.json"
)

BRATS_DIR = (
    PROJECT_ROOT
    / "Data"
    / "ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
)


MODEL_DIRS = {
    "ddpm_v5":
        EVALUATION_ROOT
        / "ddpm_v5",

    "conditional_ddpm_v3":
        EVALUATION_ROOT
        / "conditional_ddpm_v3",

    "conditional_ldm_v4":
        EVALUATION_ROOT
        / "conditional_ldm_v4"
}


MODEL_PREFIXES = {
    "ddpm_v5":
        "ddpm_v5",

    "conditional_ddpm_v3":
        "conditional_ddpm_v3",

    "conditional_ldm_v4":
        "conditional_ldm_v4"
}


CACHE_DIR = (
    PROJECT_ROOT
    / "upstream_cache"
)

SLICE_CACHE_DIR = (
    CACHE_DIR
    / "slices"
)

INCEPTION_CACHE_DIR = (
    CACHE_DIR
    / "inception_features"
)

MEDICALNET_CACHE_DIR = (
    CACHE_DIR
    / "medicalnet_features"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "upstream_results"
)


for directory in [
    REAL_DIR,
    CACHE_DIR,
    SLICE_CACHE_DIR,
    INCEPTION_CACHE_DIR,
    MEDICALNET_CACHE_DIR,
    RESULTS_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


N_VOLUMES = 200

PLANES = [
    "axial",
    "coronal",
    "sagittal"
]

# 200 volumes × 16 slices = 3,200 slices per plane.
N_SLICES_PER_PLANE = 16

INCEPTION_BATCH_SIZE = 32

MEDICALNET_BATCH_SIZE = 2

PRDC_K = 5

RUN_OPTIONAL_ASW = True

RANDOM_SEED = 2026


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "Evaluation root:",
    EVALUATION_ROOT
)

print(
    "Device:",
    DEVICE
)

In [ ]:
# ============================================================
# File and dependency checks
# ============================================================

required_paths = {
    "cohort JSON":
        COHORT_JSON,

    "data split JSON":
        DATA_SPLIT_JSON,

    "BraTS directory":
        BRATS_DIR
}


for name, path in required_paths.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Missing {name}: {path}"
        )

    print(
        f"Found {name}:",
        path
    )


optional_packages = {
    "frd_score":
        "FRD",

    "medmetric":
        "MedicalNet features",

    "SimpleITK":
        "FRD dependency"
}


import importlib.util


for package, purpose in optional_packages.items():

    available = (
        importlib.util.find_spec(
            package
        )
        is not None
    )

    print(
        f"{package}:",
        "available"
        if available
        else f"missing — required for {purpose}"
    )

In [ ]:
# ============================================================
# Load, reproduce and verify the fixed 200-subject cohort
# ============================================================

with open(
    COHORT_JSON,
    "r"
) as f:
    cohort_payload = json.load(f)


if not isinstance(
    cohort_payload,
    dict
):
    raise RuntimeError(
        "The cohort JSON must contain the cohort metadata "
        "and subject list."
    )


required_cohort_keys = {
    "cohort_size",
    "selection_seed",
    "source_sets",
    "subjects"
}


missing_cohort_keys = (
    required_cohort_keys
    - set(cohort_payload.keys())
)


if missing_cohort_keys:
    raise KeyError(
        "The cohort JSON is missing keys: "
        f"{sorted(missing_cohort_keys)}"
    )


evaluation_subjects = list(
    cohort_payload["subjects"]
)


recorded_cohort_size = int(
    cohort_payload["cohort_size"]
)


recorded_selection_seed = int(
    cohort_payload["selection_seed"]
)


recorded_source_sets = list(
    cohort_payload["source_sets"]
)


with open(
    DATA_SPLIT_JSON,
    "r"
) as f:
    split_data = json.load(f)


# Keep the original list versions because the cohort was
# sampled from an ordered held-out pool.
train_subject_list = list(
    split_data["train"]
)

validation_subject_list = list(
    split_data["validation"]
)

test_subject_list = list(
    split_data["test"]
)


train_subjects = set(
    train_subject_list
)

validation_subjects = set(
    validation_subject_list
)

test_subjects = set(
    test_subject_list
)


heldout_subjects = (
    validation_subjects
    | test_subjects
)


# This reproduces the procedure used when the cohort was
# first created during Conditional DDPM generation.
ordered_heldout_subjects = sorted(
    validation_subject_list
    + test_subject_list
)


cohort_rng = np.random.default_rng(
    recorded_selection_seed
)


expected_indices = cohort_rng.choice(
    len(ordered_heldout_subjects),
    size=recorded_cohort_size,
    replace=False
)


expected_subjects = [
    ordered_heldout_subjects[
        int(index)
    ]
    for index in expected_indices
]


# ------------------------------------------------------------
# General validation
# ------------------------------------------------------------

if recorded_cohort_size != N_VOLUMES:
    raise RuntimeError(
        "The recorded cohort size does not match "
        f"N_VOLUMES={N_VOLUMES}."
    )


if recorded_selection_seed != RANDOM_SEED:
    raise RuntimeError(
        "The cohort selection seed does not match "
        f"RANDOM_SEED={RANDOM_SEED}."
    )


if set(recorded_source_sets) != {
    "validation",
    "test"
}:
    raise RuntimeError(
        "Unexpected cohort source sets: "
        f"{recorded_source_sets}"
    )


if len(evaluation_subjects) != N_VOLUMES:
    raise RuntimeError(
        f"Expected {N_VOLUMES} subjects, "
        f"found {len(evaluation_subjects)}."
    )


if len(set(evaluation_subjects)) != N_VOLUMES:
    raise RuntimeError(
        "Duplicate subject IDs found."
    )


if not set(evaluation_subjects).issubset(
    heldout_subjects
):
    raise RuntimeError(
        "The cohort contains subjects outside "
        "the validation/test sets."
    )


if set(evaluation_subjects).intersection(
    train_subjects
):
    raise RuntimeError(
        "Training-set leakage detected."
    )


# ------------------------------------------------------------
# Reproducibility validation
# ------------------------------------------------------------

if evaluation_subjects != expected_subjects:
    raise RuntimeError(
        "The saved 200-subject cohort does not match "
        "the documented fixed-seed selection procedure."
    )


# ------------------------------------------------------------
# Save readable cohort manifest
# ------------------------------------------------------------

cohort_rows = []


for index, subject in enumerate(
    evaluation_subjects
):

    if subject in validation_subjects:
        source_split = "validation"

    elif subject in test_subjects:
        source_split = "test"

    else:
        source_split = "unknown"


    cohort_rows.append({
        "sample_id":
            f"{index:04d}",

        "subject":
            subject,

        "source_split":
            source_split,

        "selection_seed":
            recorded_selection_seed,

        "heldout_pool_size":
            len(ordered_heldout_subjects)
    })


cohort_df = pd.DataFrame(
    cohort_rows
)


cohort_manifest_path = (
    RESULTS_DIR
    / "evaluation_cohort_200.csv"
)


cohort_df.to_csv(
    cohort_manifest_path,
    index=False
)


cohort_provenance = {
    "training_subjects":
        len(train_subject_list),

    "validation_subjects":
        len(validation_subject_list),

    "test_subjects":
        len(test_subject_list),

    "heldout_pool_size":
        len(ordered_heldout_subjects),

    "selected_subjects":
        len(evaluation_subjects),

    "selection_seed":
        recorded_selection_seed,

    "sampling":
        "NumPy default_rng choice without replacement "
        "from the sorted validation-plus-test subject list",

    "cohort_json":
        str(COHORT_JSON),

    "selection_reproduced_successfully":
        True
}


cohort_provenance_path = (
    RESULTS_DIR
    / "evaluation_cohort_provenance.json"
)


with open(
    cohort_provenance_path,
    "w"
) as f:

    json.dump(
        cohort_provenance,
        f,
        indent=2
    )


print(
    "Fixed evaluation subjects:",
    len(evaluation_subjects)
)

print(
    "Held-out candidate pool:",
    len(ordered_heldout_subjects)
)

print(
    "Selection seed:",
    recorded_selection_seed
)

print(
    "Cohort selection reproduced successfully."
)

print(
    cohort_df["source_split"]
    .value_counts()
)

print(
    "Cohort manifest:",
    cohort_manifest_path
)

print(
    "Cohort provenance:",
    cohort_provenance_path
)


display(
    cohort_df.head()
)

In [ ]:
# ============================================================
# Real T2-FLAIR preprocessing
# ============================================================

def find_subject_t2f(
    subject: str
) -> Path:

    subject_dir = (
        BRATS_DIR
        / subject
    )

    if not subject_dir.is_dir():
        raise FileNotFoundError(
            f"Subject directory missing: "
            f"{subject_dir}"
        )

    candidates = [
        path
        for path in subject_dir.iterdir()
        if (
            "t2f" in path.name.lower()
            and (
                path.name.endswith(".nii")
                or path.name.endswith(".nii.gz")
            )
        )
    ]

    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one T2f file for "
            f"{subject}, found {len(candidates)}."
        )

    return candidates[0]


def preprocess_real_t2f(
    image: np.ndarray
) -> np.ndarray:

    if image.shape != (
        240,
        240,
        155
    ):
        raise ValueError(
            f"Unexpected image shape: "
            f"{image.shape}"
        )

    # Same crop used during model training.
    image = image[
        16:224,
        8:232,
        :
    ]

    # Pad depth from 155 to 160.
    image = np.pad(
        image,
        (
            (0, 0),
            (0, 0),
            (2, 3)
        ),
        mode="constant",
        constant_values=0
    )

    foreground = (
        image > 0
    )

    if not np.any(foreground):
        raise ValueError(
            "No foreground voxels found."
        )

    upper = np.percentile(
        image[foreground],
        99.9
    )

    if upper <= 0:
        raise ValueError(
            "Invalid percentile upper bound."
        )

    image = np.clip(
        image,
        0,
        upper
    )

    image = (
        image
        / upper
    )

    image[
        ~foreground
    ] = 0.0

    return image.astype(
        np.float32
    )

In [ ]:
# ============================================================
# Prepare Real 200
# ============================================================

real_metadata_rows = []


for index, subject in enumerate(
    tqdm(
        evaluation_subjects,
        desc="Preparing Real 200"
    )
):

    sample_id = (
        f"{index:04d}"
    )

    output_path = (
        REAL_DIR
        / f"real_{sample_id}.nii.gz"
    )


    if output_path.exists():

        volume = np.asarray(
            nib.load(
                output_path
            ).dataobj,
            dtype=np.float32
        )

    else:

        t2f_path = find_subject_t2f(
            subject
        )

        raw_image = nib.load(
            t2f_path
        ).get_fdata()

        volume = preprocess_real_t2f(
            raw_image
        )

        nifti = nib.Nifti1Image(
            volume,
            np.eye(
                4,
                dtype=np.float32
            )
        )

        nifti.set_data_dtype(
            np.float32
        )

        nib.save(
            nifti,
            output_path
        )


    real_metadata_rows.append({
        "sample_id":
            sample_id,

        "subject":
            subject,

        "filename":
            output_path.name,

        "shape_x":
            volume.shape[0],

        "shape_y":
            volume.shape[1],

        "shape_z":
            volume.shape[2],

        "min":
            float(volume.min()),

        "max":
            float(volume.max()),

        "mean":
            float(volume.mean()),

        "std":
            float(volume.std())
    })


real_metadata_df = pd.DataFrame(
    real_metadata_rows
)


real_metadata_path = (
    REAL_DIR
    / "metadata_real_200.csv"
)


real_metadata_df.to_csv(
    real_metadata_path,
    index=False
)


print(
    "Real volumes prepared:",
    len(real_metadata_df)
)

print(
    "Real directory:",
    REAL_DIR
)

display(
    real_metadata_df.head()
)

In [ ]:
# ============================================================
# Dataset inventory
# ============================================================

def list_nifti_files(
    directory: Path,
    prefix: str
):

    return sorted(
        directory.glob(
            f"{prefix}_*.nii.gz"
        )
    )


dataset_files = {
    "real":
        list_nifti_files(
            REAL_DIR,
            "real"
        )
}


for model_name, directory in (
    MODEL_DIRS.items()
):

    dataset_files[
        model_name
    ] = list_nifti_files(
        directory,
        MODEL_PREFIXES[
            model_name
        ]
    )


inventory_rows = []


for dataset_name, files in (
    dataset_files.items()
):

    inventory_rows.append({
        "dataset":
            dataset_name,

        "count":
            len(files),

        "complete_200":
            len(files) == N_VOLUMES
    })


inventory_df = pd.DataFrame(
    inventory_rows
)


display(
    inventory_df
)


if len(
    dataset_files["real"]
) != N_VOLUMES:

    raise RuntimeError(
        "Real reference set is not complete."
    )


EXPECTED_MODELS = list(
    MODEL_DIRS.keys()
)


incomplete_models = {
    model_name:
        len(
            dataset_files[
                model_name
            ]
        )

    for model_name in EXPECTED_MODELS

    if len(
        dataset_files[
            model_name
        ]
    ) != N_VOLUMES
}


if incomplete_models:
    raise RuntimeError(
        "One or more synthetic datasets are incomplete: "
        f"{incomplete_models}"
    )


# Use a fixed model order in all later tables and analyses.
READY_MODELS = EXPECTED_MODELS


print(
    "Models ready for upstream evaluation:",
    READY_MODELS
)


print(
    "All three synthetic datasets contain "
    f"{N_VOLUMES} complete volumes."
)

In [ ]:
# ============================================================
# Technical volume QC
# ============================================================

EXPECTED_SHAPE = (
    208,
    224,
    160
)


qc_rows = []


datasets_for_qc = [
    "real",
    *READY_MODELS
]


for dataset_name in datasets_for_qc:

    for path in tqdm(
        dataset_files[
            dataset_name
        ],
        desc=f"QC: {dataset_name}"
    ):

        volume = np.asarray(
            nib.load(
                path
            ).dataobj,
            dtype=np.float32
        )

        valid_shape = (
            volume.shape
            == EXPECTED_SHAPE
        )

        all_finite = bool(
            np.all(
                np.isfinite(
                    volume
                )
            )
        )

        within_range = bool(
            volume.min() >= -1e-6
            and volume.max() <= 1.000001
        )

        qc_rows.append({
            "dataset":
                dataset_name,

            "filename":
                path.name,

            "valid_shape":
                valid_shape,

            "all_finite":
                all_finite,

            "within_0_1":
                within_range,

            "min":
                float(volume.min()),

            "max":
                float(volume.max()),

            "mean":
                float(volume.mean()),

            "std":
                float(volume.std())
        })


qc_df = pd.DataFrame(
    qc_rows
)


qc_path = (
    RESULTS_DIR
    / "technical_qc.csv"
)


qc_df.to_csv(
    qc_path,
    index=False
)


failed_qc = qc_df[
    ~(
        qc_df["valid_shape"]
        & qc_df["all_finite"]
        & qc_df["within_0_1"]
    )
]


print(
    "QC failures:",
    len(failed_qc)
)

if len(failed_qc) > 0:
    display(failed_qc)
    raise RuntimeError(
        "Technical QC failed."
    )

print(
    "Technical QC passed."
)

In [ ]:
# ============================================================
# Persistent and resumable results storage
# ============================================================

RESULTS_JSON = (
    RESULTS_DIR
    / "upstream_results.json"
)


# Set True only when you intentionally want to recompute
# and overwrite existing metric values.
FORCE_RECOMPUTE = False


if RESULTS_JSON.exists():

    with open(
        RESULTS_JSON,
        "r"
    ) as f:

        upstream_results = json.load(f)

    print(
        "Loaded existing results:",
        RESULTS_JSON
    )

else:

    upstream_results = {}

    print(
        "No existing results file was found. "
        "A new one will be created."
    )


for model_name in MODEL_DIRS:

    upstream_results.setdefault(
        model_name,
        {}
    )


def convert_to_json_safe(
    value
):

    if isinstance(
        value,
        np.generic
    ):
        return value.item()


    if isinstance(
        value,
        np.ndarray
    ):
        return value.tolist()


    if isinstance(
        value,
        dict
    ):
        return {
            key:
                convert_to_json_safe(
                    item
                )
            for key, item in value.items()
        }


    if isinstance(
        value,
        list
    ):
        return [
            convert_to_json_safe(
                item
            )
            for item in value
        ]


    return value


def save_upstream_results():

    temporary_path = (
        RESULTS_JSON
        .with_suffix(
            ".json.tmp"
        )
    )


    safe_results = (
        convert_to_json_safe(
            upstream_results
        )
    )


    with open(
        temporary_path,
        "w"
    ) as f:

        json.dump(
            safe_results,
            f,
            indent=2,
            allow_nan=False
        )


    # Atomic replacement prevents a partially written JSON
    # if the process stops during saving.
    os.replace(
        temporary_path,
        RESULTS_JSON
    )


    print(
        "Results saved:",
        RESULTS_JSON
    )


def metric_keys_complete(
    model_name,
    required_keys
):

    model_results = (
        upstream_results
        .get(
            model_name,
            {}
        )
    )


    for key in required_keys:

        if key not in model_results:
            return False


        value = model_results[
            key
        ]


        if value is None:
            return False


        try:

            if not np.isfinite(
                float(value)
            ):
                return False

        except (
            TypeError,
            ValueError
        ):

            return False


    return True


def should_compute_metrics(
    model_name,
    required_keys,
    metric_family
):

    if FORCE_RECOMPUTE:

        print(
            f"{model_name}: recomputing "
            f"{metric_family}."
        )

        return True


    if metric_keys_complete(
        model_name,
        required_keys
    ):

        print(
            f"{model_name}: {metric_family} "
            "already saved -> skipped"
        )

        return False


    return True


save_upstream_results()

In [ ]:
# ============================================================
# Derive a fixed three-plane slice protocol from Real 200
# ============================================================

SLICE_PROTOCOL_PATH = (
    RESULTS_DIR
    / "slice_protocol.json"
)


def foreground_axis_bounds(
    volume: np.ndarray,
    axis: int
):

    other_axes = tuple(
        dimension
        for dimension in range(3)
        if dimension != axis
    )

    projection = np.any(
        volume > 1e-6,
        axis=other_axes
    )

    indices = np.where(
        projection
    )[0]

    if len(indices) == 0:
        raise RuntimeError(
            "No foreground found."
        )

    return (
        int(indices[0]),
        int(indices[-1])
    )


if SLICE_PROTOCOL_PATH.exists():

    with open(
        SLICE_PROTOCOL_PATH,
        "r"
    ) as f:
        slice_protocol = json.load(f)

    print(
        "Loaded existing slice protocol."
    )

else:

    axis_map = {
        "sagittal": 0,
        "coronal": 1,
        "axial": 2
    }

    slice_protocol = {}


    for plane, axis in axis_map.items():

        lower_bounds = []
        upper_bounds = []


        for path in tqdm(
            dataset_files["real"],
            desc=f"Bounds: {plane}"
        ):

            volume = np.asarray(
                nib.load(
                    path
                ).dataobj,
                dtype=np.float32
            )

            lower, upper = (
                foreground_axis_bounds(
                    volume,
                    axis
                )
            )

            lower_bounds.append(
                lower
            )

            upper_bounds.append(
                upper
            )


        # Use a common central region that is present
        # in the majority of real subjects.
        common_lower = int(
            np.ceil(
                np.percentile(
                    lower_bounds,
                    90
                )
            )
        )

        common_upper = int(
            np.floor(
                np.percentile(
                    upper_bounds,
                    10
                )
            )
        )


        if common_upper <= common_lower:

            common_lower = int(
                np.median(
                    lower_bounds
                )
            )

            common_upper = int(
                np.median(
                    upper_bounds
                )
            )


        indices = np.linspace(
            common_lower,
            common_upper,
            N_SLICES_PER_PLANE
        ).round().astype(int)


        indices = [
            int(index)
            for index in indices
        ]


        slice_protocol[
            plane
        ] = {
            "axis":
                axis,

            "lower":
                common_lower,

            "upper":
                common_upper,

            "indices":
                indices
        }


    with open(
        SLICE_PROTOCOL_PATH,
        "w"
    ) as f:
        json.dump(
            slice_protocol,
            f,
            indent=2
        )


print(
    json.dumps(
        slice_protocol,
        indent=2
    )
)

In [ ]:
# ============================================================
# Export fixed multi-plane slices
# ============================================================

def get_plane_slice(
    volume: np.ndarray,
    plane: str,
    index: int
):

    if plane == "axial":

        image = volume[
            :,
            :,
            index
        ]

    elif plane == "coronal":

        image = volume[
            :,
            index,
            :
        ].T

    elif plane == "sagittal":

        image = volume[
            index,
            :,
            :
        ].T

    else:

        raise ValueError(
            f"Unknown plane: {plane}"
        )

    return image


def export_dataset_slices(
    dataset_name: str,
    files: list[Path]
):

    for plane in PLANES:

        plane_dir = (
            SLICE_CACHE_DIR
            / dataset_name
            / plane
        )

        plane_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        indices = (
            slice_protocol[
                plane
            ]["indices"]
        )


        for volume_index, path in enumerate(
            tqdm(
                files,
                desc=f"{dataset_name}: {plane}"
            )
        ):

            volume = np.asarray(
                nib.load(
                    path
                ).dataobj,
                dtype=np.float32
            )

            for slice_position, index in enumerate(
                indices
            ):

                output_name = (
                    f"{volume_index:04d}"
                    f"_slice_{slice_position:02d}"
                    f"_index_{index:03d}.png"
                )

                output_path = (
                    plane_dir
                    / output_name
                )


                if output_path.exists():
                    continue


                image = get_plane_slice(
                    volume,
                    plane,
                    index
                )

                image = np.clip(
                    image,
                    0.0,
                    1.0
                )

                image_uint8 = (
                    image * 255.0
                ).round().astype(
                    np.uint8
                )

                Image.fromarray(
                    image_uint8,
                    mode="L"
                ).save(
                    output_path
                )


export_dataset_slices(
    "real",
    dataset_files["real"]
)


for model_name in READY_MODELS:

    export_dataset_slices(
        model_name,
        dataset_files[
            model_name
        ]
    )

In [ ]:
# ============================================================
# Verify slice cache
# ============================================================

EXPECTED_SLICES_PER_PLANE = (
    N_VOLUMES
    * N_SLICES_PER_PLANE
)


slice_count_rows = []


for dataset_name in [
    "real",
    *READY_MODELS
]:

    for plane in PLANES:

        plane_dir = (
            SLICE_CACHE_DIR
            / dataset_name
            / plane
        )

        count = len(
            list(
                plane_dir.glob(
                    "*.png"
                )
            )
        )

        slice_count_rows.append({
            "dataset":
                dataset_name,

            "plane":
                plane,

            "count":
                count,

            "expected":
                EXPECTED_SLICES_PER_PLANE,

            "complete":
                count
                == EXPECTED_SLICES_PER_PLANE
        })


slice_count_df = pd.DataFrame(
    slice_count_rows
)


display(
    slice_count_df
)


if not slice_count_df[
    "complete"
].all():

    raise RuntimeError(
        "Slice cache is incomplete."
    )

In [ ]:
# ============================================================
# InceptionV3 pooled and spatial features
# ============================================================

class PNGSliceDataset(Dataset):

    def __init__(
        self,
        directory: Path
    ):

        self.paths = sorted(
            directory.glob(
                "*.png"
            )
        )


    def __len__(self):

        return len(
            self.paths
        )


    def __getitem__(
        self,
        index
    ):

        path = self.paths[
            index
        ]

        image = Image.open(
            path
        ).convert(
            "L"
        )

        image = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        tensor = torch.from_numpy(
            image
        ).unsqueeze(0)

        return (
            tensor,
            path.name
        )


class InceptionPoolSpatialExtractor(
    nn.Module
):

    def __init__(self):

        super().__init__()

        weights = (
            Inception_V3_Weights.DEFAULT
        )

        model = inception_v3(
            weights=weights,
            aux_logits=True,
            transform_input=False
        )

        model.fc = nn.Identity()

        model.AuxLogits = None
        model.aux_logits = False

        model.eval()

        self.model = model

        self.spatial_output = None

        self.hook = (
            self.model.Mixed_6e
            .register_forward_hook(
                self._capture_spatial
            )
        )

        self.register_buffer(
            "mean",
            torch.tensor(
                [
                    0.485,
                    0.456,
                    0.406
                ]
            ).view(
                1,
                3,
                1,
                1
            )
        )

        self.register_buffer(
            "std",
            torch.tensor(
                [
                    0.229,
                    0.224,
                    0.225
                ]
            ).view(
                1,
                3,
                1,
                1
            )
        )


    def _capture_spatial(
        self,
        module,
        inputs,
        output
    ):

        self.spatial_output = output


    def forward(
        self,
        x
    ):

        x = x.repeat(
            1,
            3,
            1,
            1
        )

        x = F.interpolate(
            x,
            size=(
                299,
                299
            ),
            mode="bilinear",
            align_corners=False
        )

        x = (
            x - self.mean
        ) / self.std

        pooled = self.model(
            x
        )

        spatial = (
            self.spatial_output[
                :,
                :7,
                :,
                :
            ]
            .flatten(1)
        )

        return (
            pooled,
            spatial
        )


inception_extractor = (
    InceptionPoolSpatialExtractor()
    .to(DEVICE)
)

inception_extractor.eval()


print(
    "Inception extractor ready."
)

In [ ]:
# ============================================================
# Extract and cache Inception features
# ============================================================

@torch.inference_mode()
def extract_inception_features(
    slice_directory: Path,
    cache_path: Path
):

    if cache_path.exists():

        cached = np.load(
            cache_path,
            allow_pickle=True
        )

        return {
            "pool":
                cached["pool"],

            "spatial":
                cached["spatial"],

            "filenames":
                cached["filenames"]
        }


    dataset = PNGSliceDataset(
        slice_directory
    )

    loader = DataLoader(
        dataset,
        batch_size=INCEPTION_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=(
            DEVICE.type
            == "cuda"
        )
    )


    pool_features = []
    spatial_features = []
    filenames = []


    for batch, batch_names in tqdm(
        loader,
        desc=(
            "Inception: "
            f"{slice_directory}"
        )
    ):

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )

        pool, spatial = (
            inception_extractor(
                batch
            )
        )

        pool_features.append(
            pool.cpu().numpy()
        )

        spatial_features.append(
            spatial.cpu().numpy()
        )

        filenames.extend(
            list(batch_names)
        )


    pool_features = np.concatenate(
        pool_features,
        axis=0
    ).astype(
        np.float32
    )

    spatial_features = np.concatenate(
        spatial_features,
        axis=0
    ).astype(
        np.float32
    )

    filenames = np.asarray(
        filenames
    )


    cache_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    np.savez_compressed(
        cache_path,
        pool=pool_features,
        spatial=spatial_features,
        filenames=filenames
    )


    return {
        "pool":
            pool_features,

        "spatial":
            spatial_features,

        "filenames":
            filenames
    }


inception_features = {}


for dataset_name in [
    "real",
    *READY_MODELS
]:

    inception_features[
        dataset_name
    ] = {}


    for plane in PLANES:

        slice_directory = (
            SLICE_CACHE_DIR
            / dataset_name
            / plane
        )

        cache_path = (
            INCEPTION_CACHE_DIR
            / f"{dataset_name}_{plane}.npz"
        )

        inception_features[
            dataset_name
        ][
            plane
        ] = extract_inception_features(
            slice_directory,
            cache_path
        )


print(
    "Inception feature extraction complete."
)

In [ ]:
# ============================================================
# Distribution metric functions
# ============================================================

def frechet_distance(
    features_a: np.ndarray,
    features_b: np.ndarray,
    eps: float = 1e-6
) -> float:

    features_a = np.asarray(
        features_a,
        dtype=np.float64
    )

    features_b = np.asarray(
        features_b,
        dtype=np.float64
    )


    mean_a = features_a.mean(
        axis=0
    )

    mean_b = features_b.mean(
        axis=0
    )


    covariance_a = np.cov(
        features_a,
        rowvar=False
    )

    covariance_b = np.cov(
        features_b,
        rowvar=False
    )


    difference = (
        mean_a - mean_b
    )


    covariance_product = (
        covariance_a
        @ covariance_b
    )


    covariance_mean = linalg.sqrtm(
        covariance_product
    )


    if not np.isfinite(
        covariance_mean
    ).all():

        offset = (
            np.eye(
                covariance_a.shape[0]
            )
            * eps
        )

        covariance_mean = linalg.sqrtm(
            (
                covariance_a
                + offset
            )
            @ (
                covariance_b
                + offset
            )
        )


    if np.iscomplexobj(
        covariance_mean
    ):

        covariance_mean = (
            covariance_mean.real
        )


    score = (
        difference.dot(
            difference
        )
        + np.trace(
            covariance_a
            + covariance_b
            - 2.0
            * covariance_mean
        )
    )


    return float(
        np.real(
            score
        )
    )


def polynomial_kernel(
    x,
    y
):

    dimension = (
        x.shape[1]
    )

    return (
        (
            x @ y.T
            / dimension
        )
        + 1.0
    ) ** 3


def kid_unbiased_once(
    features_a,
    features_b
):

    x = np.asarray(
        features_a,
        dtype=np.float64
    )

    y = np.asarray(
        features_b,
        dtype=np.float64
    )


    m = x.shape[0]
    n = y.shape[0]


    if m < 2 or n < 2:
        raise ValueError(
            "KID requires at least two samples."
        )


    kernel_xx = polynomial_kernel(
        x,
        x
    )

    kernel_yy = polynomial_kernel(
        y,
        y
    )

    kernel_xy = polynomial_kernel(
        x,
        y
    )


    term_xx = (
        (
            kernel_xx.sum()
            - np.trace(
                kernel_xx
            )
        )
        / (
            m
            * (m - 1)
        )
    )


    term_yy = (
        (
            kernel_yy.sum()
            - np.trace(
                kernel_yy
            )
        )
        / (
            n
            * (n - 1)
        )
    )


    term_xy = (
        2.0
        * kernel_xy.mean()
    )


    return float(
        term_xx
        + term_yy
        - term_xy
    )


def kid_score(
    features_a,
    features_b,
    subset_size=1000,
    repeats=20,
    seed=2026
):

    rng = np.random.default_rng(
        seed
    )


    subset_size = min(
        subset_size,
        len(features_a),
        len(features_b)
    )


    values = []


    for _ in range(repeats):

        index_a = rng.choice(
            len(features_a),
            size=subset_size,
            replace=False
        )

        index_b = rng.choice(
            len(features_b),
            size=subset_size,
            replace=False
        )


        values.append(
            kid_unbiased_once(
                features_a[
                    index_a
                ],
                features_b[
                    index_b
                ]
            )
        )


    return {
        "mean":
            float(
                np.mean(
                    values
                )
            ),

        "std":
            float(
                np.std(
                    values,
                    ddof=1
                )
            )
    }

In [ ]:
# ============================================================
# Compute FID, KID and sFID
# ============================================================

FID_FAMILY_KEYS = (
    [
        "FID",
        "KID",
        "sFID"
    ]
    +
    [
        f"FID_{plane}"
        for plane in PLANES
    ]
    +
    [
        f"KID_{plane}"
        for plane in PLANES
    ]
    +
    [
        f"KID_{plane}_std"
        for plane in PLANES
    ]
    +
    [
        f"sFID_{plane}"
        for plane in PLANES
    ]
)


for model_name in READY_MODELS:

    if not should_compute_metrics(
        model_name,
        FID_FAMILY_KEYS,
        "FID/KID/sFID"
    ):
        continue


    fid_plane_scores = {}
    kid_plane_scores = {}
    sfid_plane_scores = {}


    for plane in PLANES:

        real_pool = (
            inception_features[
                "real"
            ][
                plane
            ][
                "pool"
            ]
        )

        synthetic_pool = (
            inception_features[
                model_name
            ][
                plane
            ][
                "pool"
            ]
        )


        real_spatial = (
            inception_features[
                "real"
            ][
                plane
            ][
                "spatial"
            ]
        )

        synthetic_spatial = (
            inception_features[
                model_name
            ][
                plane
            ][
                "spatial"
            ]
        )


        fid_value = frechet_distance(
            real_pool,
            synthetic_pool
        )


        kid_values = kid_score(
            real_pool,
            synthetic_pool,
            subset_size=1000,
            repeats=20,
            seed=RANDOM_SEED
        )


        sfid_value = frechet_distance(
            real_spatial,
            synthetic_spatial
        )


        fid_plane_scores[
            plane
        ] = fid_value

        kid_plane_scores[
            plane
        ] = kid_values["mean"]

        sfid_plane_scores[
            plane
        ] = sfid_value


        upstream_results[
            model_name
        ][
            f"FID_{plane}"
        ] = fid_value


        upstream_results[
            model_name
        ][
            f"KID_{plane}"
        ] = kid_values["mean"]


        upstream_results[
            model_name
        ][
            f"KID_{plane}_std"
        ] = kid_values["std"]


        upstream_results[
            model_name
        ][
            f"sFID_{plane}"
        ] = sfid_value


    upstream_results[
        model_name
    ][
        "FID"
    ] = float(
        np.mean(
            list(
                fid_plane_scores.values()
            )
        )
    )


    upstream_results[
        model_name
    ][
        "KID"
    ] = float(
        np.mean(
            list(
                kid_plane_scores.values()
            )
        )
    )


    upstream_results[
        model_name
    ][
        "sFID"
    ] = float(
        np.mean(
            list(
                sfid_plane_scores.values()
            )
        )
    )


    print()
    print(
        model_name
    )

    print(
        "FID:",
        upstream_results[
            model_name
        ]["FID"]
    )

    print(
        "KID:",
        upstream_results[
            model_name
        ]["KID"]
    )

    print(
        "sFID:",
        upstream_results[
            model_name
        ]["sFID"]
    )


    # Save immediately after each model.
    save_upstream_results()

In [ ]:
# ============================================================
# RadFID external implementation
# ============================================================

RADFID_REPO = (
    PROJECT_ROOT
    / "third_party"
    / "medical-image-similarity-metrics"
)


RADFID_SCRIPT = (
    RADFID_REPO
    / "compute_allmetrics.sh"
)


RADFID_WEIGHT = (
    RADFID_REPO
    / "src"
    / "gan-metrics-pytorch"
    / "models"
    / "RadImageNet_InceptionV3.pt"
)


RADFID_READY = (
    RADFID_SCRIPT.exists()
    and RADFID_WEIGHT.exists()
)


print(
    "RadFID repository:",
    RADFID_REPO
)

print(
    "RadFID ready:",
    RADFID_READY
)


if not RADFID_READY:

    print(
        "RadFID is temporarily skipped. "
        "The repository and RadImageNet "
        "InceptionV3 weights must be uploaded."
    )

In [ ]:
# ============================================================
# Compute RadFID
# ============================================================

def parse_radfid_output(
    output: str
) -> float:

    patterns = [
        r"RadFID[^0-9+\-]*"
        r"([+\-]?"
        r"\d+(?:\.\d+)?"
        r"(?:[eE][+\-]?\d+)?)",

        r"radfid[^0-9+\-]*"
        r"([+\-]?"
        r"\d+(?:\.\d+)?"
        r"(?:[eE][+\-]?\d+)?)"
    ]


    for pattern in patterns:

        matches = re.findall(
            pattern,
            output,
            flags=re.IGNORECASE
        )

        if matches:
            return float(
                matches[-1]
            )


    raise RuntimeError(
        "Could not parse RadFID output:\n"
        + output
    )


def run_radfid(
    real_slice_dir: Path,
    synthetic_slice_dir: Path
):

    command = [
        "bash",
        str(
            RADFID_SCRIPT
        ),
        str(
            real_slice_dir.resolve()
        ),
        str(
            synthetic_slice_dir.resolve()
        ),
        "RadFID"
    ]


    process = subprocess.run(
        command,
        cwd=RADFID_REPO,
        capture_output=True,
        text=True,
        check=True
    )


    print(
        process.stdout
    )


    return parse_radfid_output(
        process.stdout
        + "\n"
        + process.stderr
    )


RADFID_KEYS = (
    [
        "RadFID"
    ]
    +
    [
        f"RadFID_{plane}"
        for plane in PLANES
    ]
)


if RADFID_READY:

    for model_name in READY_MODELS:

        if not should_compute_metrics(
            model_name,
            RADFID_KEYS,
            "RadFID"
        ):
            continue


        plane_scores = {}


        for plane in PLANES:

            value = run_radfid(
                SLICE_CACHE_DIR
                / "real"
                / plane,

                SLICE_CACHE_DIR
                / model_name
                / plane
            )


            plane_scores[
                plane
            ] = value


            upstream_results[
                model_name
            ][
                f"RadFID_{plane}"
            ] = value


        upstream_results[
            model_name
        ][
            "RadFID"
        ] = float(
            np.mean(
                list(
                    plane_scores.values()
                )
            )
        )


        print(
            model_name,
            "RadFID:",
            upstream_results[
                model_name
            ][
                "RadFID"
            ]
        )


        save_upstream_results()

else:

    print(
        "RadFID skipped because the external repository "
        "or RadImageNet weights are unavailable."
    )

In [ ]:
# ============================================================
# Load MedicalNet ResNet50 feature extractor
# ============================================================

try:

    from medmetric.extractors.medicalnet import (
        MedicalNetFeatureExtractor
    )

except ImportError as error:

    raise ImportError(
        "medmetric is required for the "
        "3D MedicalNet metrics. "
        "Install it in python_env before "
        "running this section."
    ) from error


medicalnet_extractor = (
    MedicalNetFeatureExtractor
    .from_pretrained(
        depth=50,
        use_23dataset=True,
        device=str(DEVICE)
    )
)


medicalnet_extractor.eval()


print(
    "MedicalNet ResNet50 ready."
)

In [ ]:
# ============================================================
# 3D MedicalNet datasets
# ============================================================

MEDICALNET_SIZE = (
    112,
    112,
    112
)


def medicalnet_preprocess(
    volume: np.ndarray
):

    volume = np.asarray(
        volume,
        dtype=np.float32
    )

    volume = np.clip(
        volume,
        0.0,
        1.0
    )

    # Convert [X,Y,Z] to [D,H,W].
    volume = np.transpose(
        volume,
        (
            2,
            1,
            0
        )
    )


    tensor = torch.from_numpy(
        volume
    ).unsqueeze(0).unsqueeze(0)


    tensor = F.interpolate(
        tensor,
        size=MEDICALNET_SIZE,
        mode="trilinear",
        align_corners=False
    )


    spatial_mean = tensor.mean(
        dim=(
            2,
            3,
            4
        ),
        keepdim=True
    )


    spatial_std = tensor.std(
        dim=(
            2,
            3,
            4
        ),
        keepdim=True
    ).clamp_min(
        1e-8
    )


    tensor = (
        tensor
        - spatial_mean
    ) / spatial_std


    return tensor.squeeze(0)


class PreparedNiftiMedicalNetDataset(
    Dataset
):

    def __init__(
        self,
        paths
    ):

        self.paths = list(
            paths
        )


    def __len__(self):

        return len(
            self.paths
        )


    def __getitem__(
        self,
        index
    ):

        path = self.paths[
            index
        ]

        volume = np.asarray(
            nib.load(
                path
            ).dataobj,
            dtype=np.float32
        )


        return (
            medicalnet_preprocess(
                volume
            ),
            path.name
        )


class RawBraTSMedicalNetDataset(
    Dataset
):

    def __init__(
        self,
        subjects
    ):

        self.subjects = list(
            subjects
        )


    def __len__(self):

        return len(
            self.subjects
        )


    def __getitem__(
        self,
        index
    ):

        subject = self.subjects[
            index
        ]

        path = find_subject_t2f(
            subject
        )

        raw = nib.load(
            path
        ).get_fdata()

        volume = preprocess_real_t2f(
            raw
        )


        return (
            medicalnet_preprocess(
                volume
            ),
            subject
        )

In [ ]:
# ============================================================
# Extract and cache MedicalNet features
# ============================================================

@torch.inference_mode()
def extract_medicalnet_features(
    dataset: Dataset,
    cache_path: Path
):

    if cache_path.exists():

        cached = np.load(
            cache_path,
            allow_pickle=True
        )

        return {
            "features":
                cached["features"],

            "identifiers":
                cached["identifiers"]
        }


    loader = DataLoader(
        dataset,
        batch_size=MEDICALNET_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=(
            DEVICE.type
            == "cuda"
        )
    )


    all_features = []
    identifiers = []


    for volumes, batch_ids in tqdm(
        loader,
        desc=f"MedicalNet: {cache_path.stem}"
    ):

        volumes = volumes.to(
            DEVICE,
            non_blocking=True
        )


        features = medicalnet_extractor(
            volumes
        )


        if isinstance(
            features,
            (
                tuple,
                list
            )
        ):
            features = features[0]


        features = features.flatten(
            start_dim=1
        )


        all_features.append(
            features
            .detach()
            .cpu()
            .numpy()
        )


        identifiers.extend(
            list(batch_ids)
        )


    all_features = np.concatenate(
        all_features,
        axis=0
    ).astype(
        np.float32
    )


    identifiers = np.asarray(
        identifiers
    )


    cache_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    np.savez_compressed(
        cache_path,
        features=all_features,
        identifiers=identifiers
    )


    return {
        "features":
            all_features,

        "identifiers":
            identifiers
    }


# Training features are needed for AuthPct.
training_medicalnet = (
    extract_medicalnet_features(
        RawBraTSMedicalNetDataset(
            split_data["train"]
        ),
        MEDICALNET_CACHE_DIR
        / "training_1000.npz"
    )
)


real_medicalnet = (
    extract_medicalnet_features(
        PreparedNiftiMedicalNetDataset(
            dataset_files[
                "real"
            ]
        ),
        MEDICALNET_CACHE_DIR
        / "real_200.npz"
    )
)


synthetic_medicalnet = {}


for model_name in READY_MODELS:

    synthetic_medicalnet[
        model_name
    ] = extract_medicalnet_features(
        PreparedNiftiMedicalNetDataset(
            dataset_files[
                model_name
            ]
        ),
        MEDICALNET_CACHE_DIR
        / f"{model_name}_200.npz"
    )


print(
    "Training MedicalNet features:",
    training_medicalnet[
        "features"
    ].shape
)

print(
    "Real MedicalNet features:",
    real_medicalnet[
        "features"
    ].shape
)

In [ ]:
# ============================================================
# MedicalNet feature normalization
# ============================================================

train_med_raw = (
    training_medicalnet[
        "features"
    ]
)

real_med_raw = (
    real_medicalnet[
        "features"
    ]
)


synthetic_med_raw = {
    model_name:
        synthetic_medicalnet[
            model_name
        ][
            "features"
        ]

    for model_name in READY_MODELS
}


medicalnet_scaler = (
    StandardScaler()
    .fit(
        train_med_raw
    )
)


train_med_scaled = (
    medicalnet_scaler
    .transform(
        train_med_raw
    )
    .astype(
        np.float32
    )
)


real_med_scaled = (
    medicalnet_scaler
    .transform(
        real_med_raw
    )
    .astype(
        np.float32
    )
)


synthetic_med_scaled = {
    model_name:
        medicalnet_scaler
        .transform(
            features
        )
        .astype(
            np.float32
        )

    for model_name, features in (
        synthetic_med_raw.items()
    )
}


np.savez_compressed(
    MEDICALNET_CACHE_DIR
    / "medicalnet_scaler.npz",
    mean=medicalnet_scaler.mean_,
    scale=medicalnet_scaler.scale_
)


print(
    "MedicalNet raw feature dimension:",
    real_med_raw.shape[1]
)

In [ ]:
# ============================================================
# 3D MedicalNet-based Fréchet Distance
# ============================================================

for model_name in READY_MODELS:

    if not should_compute_metrics(
        model_name,
        [
            "MedFID"
        ],
        "MedFID"
    ):
        continue


    medfid = frechet_distance(
        real_med_raw,
        synthetic_med_raw[
            model_name
        ]
    )


    upstream_results[
        model_name
    ][
        "MedFID"
    ] = medfid


    print(
        model_name,
        "MedFID:",
        medfid
    )


    save_upstream_results()

In [ ]:
# ============================================================
# PRDC
# ============================================================

def nearest_neighbour_radius(
    features,
    k=5
):

    distances = cdist(
        features,
        features,
        metric="euclidean"
    )

    sorted_distances = np.sort(
        distances,
        axis=1
    )

    # Column zero is the distance to the sample itself.
    return sorted_distances[
        :,
        k
    ]


def compute_prdc(
    real_features,
    fake_features,
    k=5
):

    real_features = np.asarray(
        real_features,
        dtype=np.float64
    )

    fake_features = np.asarray(
        fake_features,
        dtype=np.float64
    )


    if len(real_features) <= k:
        raise ValueError(
            "The number of real samples must exceed k."
        )


    if len(fake_features) <= k:
        raise ValueError(
            "The number of synthetic samples must exceed k."
        )


    real_radius = (
        nearest_neighbour_radius(
            real_features,
            k=k
        )
    )

    fake_radius = (
        nearest_neighbour_radius(
            fake_features,
            k=k
        )
    )


    real_fake_distances = cdist(
        real_features,
        fake_features,
        metric="euclidean"
    )


    precision = np.mean(
        np.any(
            real_fake_distances
            <= real_radius[
                :,
                None
            ],
            axis=0
        )
    )


    recall = np.mean(
        np.any(
            real_fake_distances
            <= fake_radius[
                None,
                :
            ],
            axis=1
        )
    )


    density = np.mean(
        np.sum(
            real_fake_distances
            <= real_radius[
                :,
                None
            ],
            axis=0
        )
        / k
    )


    coverage = np.mean(
        np.min(
            real_fake_distances,
            axis=1
        )
        <= real_radius
    )


    return {
        "Precision":
            float(precision),

        "Recall":
            float(recall),

        "Density":
            float(density),

        "Coverage":
            float(coverage)
    }


PRDC_KEYS = [
    "Precision",
    "Recall",
    "Density",
    "Coverage"
]


for model_name in READY_MODELS:

    if not should_compute_metrics(
        model_name,
        PRDC_KEYS,
        "PRDC"
    ):
        continue


    prdc = compute_prdc(
        real_med_scaled,
        synthetic_med_scaled[
            model_name
        ],
        k=PRDC_K
    )


    upstream_results[
        model_name
    ].update(
        prdc
    )


    print(
        model_name,
        prdc
    )


    save_upstream_results()

In [ ]:
# ============================================================
# Vendi Score
# ============================================================

def squared_distance_matrix(
    features
):

    return cdist(
        features,
        features,
        metric="sqeuclidean"
    )


real_squared_distances = (
    squared_distance_matrix(
        real_med_scaled
    )
)


positive_distances = (
    real_squared_distances[
        real_squared_distances > 0
    ]
)


if len(positive_distances) == 0:
    raise RuntimeError(
        "No positive pairwise distances were found "
        "for the real reference features."
    )


RBF_SIGMA_SQUARED = float(
    np.median(
        positive_distances
    )
)


if RBF_SIGMA_SQUARED <= 0:
    raise RuntimeError(
        "Invalid RBF bandwidth."
    )


def rbf_similarity_matrix(
    features,
    sigma_squared
):

    distances = (
        squared_distance_matrix(
            features
        )
    )

    kernel = np.exp(
        -distances
        / (
            2.0
            * sigma_squared
        )
    )

    np.fill_diagonal(
        kernel,
        1.0
    )

    return kernel


def vendi_score_from_kernel(
    kernel
):

    kernel = np.asarray(
        kernel,
        dtype=np.float64
    )


    eigenvalues = np.linalg.eigvalsh(
        kernel
        / kernel.shape[0]
    )


    eigenvalues = np.clip(
        eigenvalues,
        0.0,
        None
    )


    total = eigenvalues.sum()


    if total <= 0:
        raise RuntimeError(
            "Invalid Vendi eigenvalues."
        )


    eigenvalues = (
        eigenvalues
        / total
    )


    positive = eigenvalues[
        eigenvalues > 0
    ]


    entropy = -np.sum(
        positive
        * np.log(
            positive
        )
    )


    return float(
        np.exp(
            entropy
        )
    )


real_vendi = (
    vendi_score_from_kernel(
        rbf_similarity_matrix(
            real_med_scaled,
            RBF_SIGMA_SQUARED
        )
    )
)


print(
    "Real Vendi Score:",
    real_vendi
)


VENDI_KEYS = [
    "Vendi",
    "Vendi_real_reference",
    "Vendi_ratio_to_real"
]


for model_name in READY_MODELS:

    if not should_compute_metrics(
        model_name,
        VENDI_KEYS,
        "Vendi Score"
    ):
        continue


    synthetic_vendi = (
        vendi_score_from_kernel(
            rbf_similarity_matrix(
                synthetic_med_scaled[
                    model_name
                ],
                RBF_SIGMA_SQUARED
            )
        )
    )


    upstream_results[
        model_name
    ][
        "Vendi"
    ] = synthetic_vendi


    upstream_results[
        model_name
    ][
        "Vendi_real_reference"
    ] = real_vendi


    upstream_results[
        model_name
    ][
        "Vendi_ratio_to_real"
    ] = float(
        synthetic_vendi
        / real_vendi
    )


    print(
        model_name,
        "Vendi:",
        synthetic_vendi,
        "| ratio to real:",
        upstream_results[
            model_name
        ][
            "Vendi_ratio_to_real"
        ]
    )


    save_upstream_results()

In [ ]:
# ============================================================
# Authenticity Percentage
# ============================================================

def compute_training_nn_radius(
    training_features
):

    training_features = np.asarray(
        training_features,
        dtype=np.float64
    )


    train_train_distances = cdist(
        training_features,
        training_features,
        metric="euclidean"
    )


    np.fill_diagonal(
        train_train_distances,
        np.inf
    )


    return np.min(
        train_train_distances,
        axis=1
    )


def compute_authenticity_percentage(
    training_features,
    synthetic_features,
    training_nn_radius
):

    training_features = np.asarray(
        training_features,
        dtype=np.float64
    )

    synthetic_features = np.asarray(
        synthetic_features,
        dtype=np.float64
    )

    training_nn_radius = np.asarray(
        training_nn_radius,
        dtype=np.float64
    )


    synthetic_train_distances = cdist(
        synthetic_features,
        training_features,
        metric="euclidean"
    )


    nearest_training_index = np.argmin(
        synthetic_train_distances,
        axis=1
    )


    synthetic_nearest_distance = np.min(
        synthetic_train_distances,
        axis=1
    )


    corresponding_training_radius = (
        training_nn_radius[
            nearest_training_index
        ]
    )


    authentic = (
        synthetic_nearest_distance
        > corresponding_training_radius
    )


    return {
        "AuthPct":
            float(
                authentic.mean()
                * 100.0
            ),

        "authentic_flags":
            authentic,

        "nearest_training_index":
            nearest_training_index,

        "synthetic_nearest_distance":
            synthetic_nearest_distance,

        "training_radius":
            corresponding_training_radius
    }


# This only needs to be calculated once.
training_nn_radius = compute_training_nn_radius(
    train_med_scaled
)


training_identifiers = np.asarray(
    training_medicalnet[
        "identifiers"
    ]
)


for model_name in READY_MODELS:

    audit_path = (
        RESULTS_DIR
        / f"{model_name}_authenticity_audit.csv"
    )


    authenticity_already_complete = (
        metric_keys_complete(
            model_name,
            [
                "AuthPct"
            ]
        )
        and audit_path.exists()
    )


    if (
        authenticity_already_complete
        and not FORCE_RECOMPUTE
    ):

        print(
            f"{model_name}: AuthPct and audit CSV "
            "already saved -> skipped"
        )

        continue


    authenticity = (
        compute_authenticity_percentage(
            train_med_scaled,
            synthetic_med_scaled[
                model_name
            ],
            training_nn_radius
        )
    )


    upstream_results[
        model_name
    ][
        "AuthPct"
    ] = authenticity[
        "AuthPct"
    ]


    synthetic_identifiers = np.asarray(
        synthetic_medicalnet[
            model_name
        ][
            "identifiers"
        ]
    )


    nearest_training_index = (
        authenticity[
            "nearest_training_index"
        ]
    )


    pd.DataFrame({
        "synthetic_index":
            np.arange(
                len(
                    synthetic_identifiers
                )
            ),

        "synthetic_identifier":
            synthetic_identifiers,

        "authentic":
            authenticity[
                "authentic_flags"
            ],

        "nearest_training_index":
            nearest_training_index,

        "nearest_training_identifier":
            training_identifiers[
                nearest_training_index
            ],

        "synthetic_nearest_distance":
            authenticity[
                "synthetic_nearest_distance"
            ],

        "training_nn_radius":
            authenticity[
                "training_radius"
            ]
    }).to_csv(
        audit_path,
        index=False
    )


    print(
        model_name,
        "AuthPct:",
        authenticity[
            "AuthPct"
        ]
    )

    print(
        "Authenticity audit saved:",
        audit_path
    )


    save_upstream_results()

In [ ]:
# ============================================================
# Approximate Sliced-Wasserstein Distance
# ============================================================

def sliced_wasserstein_once(
    features_a,
    features_b,
    n_projections=1024,
    seed=2026
):

    x = np.asarray(
        features_a,
        dtype=np.float64
    )

    y = np.asarray(
        features_b,
        dtype=np.float64
    )


    if len(x) != len(y):
        raise ValueError(
            "This implementation expects "
            "equal sample counts."
        )


    rng = np.random.default_rng(
        seed
    )


    directions = rng.normal(
        size=(
            x.shape[1],
            n_projections
        )
    )


    directions /= np.linalg.norm(
        directions,
        axis=0,
        keepdims=True
    ).clip(
        min=1e-12
    )


    projected_x = (
        x @ directions
    )

    projected_y = (
        y @ directions
    )


    projected_x.sort(
        axis=0
    )

    projected_y.sort(
        axis=0
    )


    wasserstein_per_projection = np.mean(
        np.abs(
            projected_x
            - projected_y
        ),
        axis=0
    )


    return float(
        np.mean(
            wasserstein_per_projection
        )
    )


def approximate_sliced_wasserstein(
    features_a,
    features_b,
    n_projections=1024,
    repeats=5,
    base_seed=2026
):

    values = []


    for repeat in range(
        repeats
    ):

        values.append(
            sliced_wasserstein_once(
                features_a,
                features_b,
                n_projections=n_projections,
                seed=(
                    base_seed
                    + repeat
                )
            )
        )


    return {
        "mean":
            float(
                np.mean(
                    values
                )
            ),

        "std":
            float(
                np.std(
                    values,
                    ddof=1
                )
            )
    }


if RUN_OPTIONAL_ASW:

    ASW_KEYS = [
        "ASW",
        "ASW_std"
    ]


    for model_name in READY_MODELS:

        if not should_compute_metrics(
            model_name,
            ASW_KEYS,
            "ASW"
        ):
            continue


        asw = (
            approximate_sliced_wasserstein(
                real_med_scaled,
                synthetic_med_scaled[
                    model_name
                ],
                n_projections=1024,
                repeats=5,
                base_seed=RANDOM_SEED
            )
        )


        upstream_results[
            model_name
        ][
            "ASW"
        ] = asw[
            "mean"
        ]


        upstream_results[
            model_name
        ][
            "ASW_std"
        ] = asw[
            "std"
        ]


        print(
            model_name,
            "ASW:",
            asw
        )


        save_upstream_results()

else:

    print(
        "ASW is disabled by RUN_OPTIONAL_ASW=False."
    )

In [ ]:
# ============================================================
# Fréchet Radiomic Distance
# ============================================================

print(
    "FRD is evaluated separately in FRD_Upstreaming.ipynb "
    "with one CPU worker to control system-memory usage."
)


In [ ]:
# ============================================================
# Main upstream comparison table
# ============================================================

MAIN_METRICS = [
    "FID",
    "KID",
    "sFID",
    "Precision",
    "Recall",
    "Density",
    "Coverage",
    "FRD",
    "RadFID",
    "MedFID",
    "Vendi",
    "AuthPct",
    "ASW"
]


comparison_rows = []


for model_name in MODEL_DIRS:

    row = {
        "Model":
            model_name
    }


    for metric in MAIN_METRICS:

        row[
            metric
        ] = (
            upstream_results
            .get(
                model_name,
                {}
            )
            .get(
                metric,
                np.nan
            )
        )


    comparison_rows.append(
        row
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


comparison_path = (
    RESULTS_DIR
    / "upstream_comparison.csv"
)


comparison_df.to_csv(
    comparison_path,
    index=False
)


display(
    comparison_df
)


print(
    "Comparison table saved:",
    comparison_path
)

In [ ]:
# ============================================================
# Plane-specific results
# ============================================================

plane_rows = []


for model_name in MODEL_DIRS:

    model_results = (
        upstream_results.get(
            model_name,
            {}
        )
    )


    for plane in PLANES:

        plane_rows.append({
            "Model":
                model_name,

            "Plane":
                plane,

            "FID":
                model_results.get(
                    f"FID_{plane}",
                    np.nan
                ),

            "KID":
                model_results.get(
                    f"KID_{plane}",
                    np.nan
                ),

            "sFID":
                model_results.get(
                    f"sFID_{plane}",
                    np.nan
                ),

            "RadFID":
                model_results.get(
                    f"RadFID_{plane}",
                    np.nan
                )
        })


plane_results_df = pd.DataFrame(
    plane_rows
)


plane_results_path = (
    RESULTS_DIR
    / "upstream_plane_results.csv"
)


plane_results_df.to_csv(
    plane_results_path,
    index=False
)


display(
    plane_results_df
)

In [ ]:
# ============================================================
# Save reproducible evaluation protocol
# ============================================================

evaluation_protocol = {
    "dataset":
        "BraTS 2023 GLI",

    "modality":
        "T2-FLAIR",

    "volume_shape": [
        208,
        224,
        160
    ],

    "evaluation_cohort_size":
        N_VOLUMES,

    "cohort_source":
        "held-out validation and test subjects",

    "cohort_json":
        str(COHORT_JSON),

    "real_intensity_range": [
        0.0,
        1.0
    ],

    "slices_per_plane":
        N_SLICES_PER_PLANE,

    "planes":
        PLANES,

    "metrics": {
        "FID":
            "2D ImageNet InceptionV3 pooled features",

        "KID":
            "2D ImageNet InceptionV3 pooled features",

        "sFID":
            "first seven channels of InceptionV3 Mixed_6e features",

        "PRDC":
            "standardised 3D MedicalNet ResNet50 features",

        "FRD":
            "FRDv1 on complete 3D NIfTI volumes",

        "RadFID":
            "2D RadImageNet InceptionV3 features",

        "MedFID":
            "3D MedicalNet ResNet50 pooled features",

        "Vendi":
            "RBF kernel over standardised MedicalNet features",

        "AuthPct":
            "MedicalNet features compared with the 1000-subject generative training set",

        "ASW":
            "random projections over standardised MedicalNet features"
    },

    "metric_direction": {
        "FID":
            "lower",

        "KID":
            "lower",

        "sFID":
            "lower",

        "Precision":
            "higher",

        "Recall":
            "higher",

        "Density":
            "higher",

        "Coverage":
            "higher",

        "FRD":
            "lower",

        "RadFID":
            "lower",

        "MedFID":
            "lower",

        "Vendi":
            "higher, interpreted relative to real reference",

        "AuthPct":
            "higher",

        "ASW":
            "lower"
    },

    "random_seed":
        RANDOM_SEED
}


protocol_path = (
    RESULTS_DIR
    / "upstream_evaluation_protocol.json"
)


with open(
    protocol_path,
    "w"
) as f:

    json.dump(
        evaluation_protocol,
        f,
        indent=2
    )


print(
    "Evaluation protocol saved:",
    protocol_path
)

In [ ]:
# ============================================================
# Final metric completeness audit
# ============================================================

REQUIRED_FINAL_METRICS = [
    "FID",
    "KID",
    "sFID",
    "Precision",
    "Recall",
    "Density",
    "Coverage",
    "FRD",
    "RadFID",
    "MedFID",
    "Vendi",
    "AuthPct"
]


if RUN_OPTIONAL_ASW:

    REQUIRED_FINAL_METRICS.append(
        "ASW"
    )


completion_rows = []


for model_name in READY_MODELS:

    model_results = (
        upstream_results.get(
            model_name,
            {}
        )
    )


    missing_metrics = []


    for metric in REQUIRED_FINAL_METRICS:

        if metric not in model_results:

            missing_metrics.append(
                metric
            )

            continue


        value = model_results[
            metric
        ]


        try:

            value_is_valid = np.isfinite(
                float(value)
            )

        except (
            TypeError,
            ValueError
        ):

            value_is_valid = False


        if not value_is_valid:

            missing_metrics.append(
                metric
            )


    completion_rows.append({
        "Model":
            model_name,

        "Complete":
            len(missing_metrics) == 0,

        "Missing metrics":
            ", ".join(
                missing_metrics
            )
            if missing_metrics
            else ""
    })


completion_df = pd.DataFrame(
    completion_rows
)


display(
    completion_df
)


completion_path = (
    RESULTS_DIR
    / "upstream_metric_completeness.csv"
)


completion_df.to_csv(
    completion_path,
    index=False
)


if completion_df[
    "Complete"
].all():

    print(
        "All requested upstream metrics are complete "
        "for all three models."
    )

else:

    print(
        "Some metrics remain incomplete. "
        "This is expected if RadFID resources or "
        "other optional external dependencies are unavailable."
    )


print(
    "Completeness audit saved:",
    completion_path
)